In [20]:
import numpy as np
import pandas as pd
import cupy as cp
import cupyx
import tqdm
import torch
from typing import Dict
from utils import utils
from utils.data_utils import load_metadata_and_map

In [21]:
METADATA_FILE = "embeddings/id_metadata_arsmagna_test.json"
TENSOR_DIR = "embeddings/uuid_embeddings/"

In [22]:
N_CLUSTERS = 64
N_PCA_COMPONENTS = 256

In [23]:
# 1. Load Tensors
tensors = utils.load_tensor(TENSOR_DIR, num_workers=16)


Loading cache from embeddings/cache
Cache file embeddings/cache/6aadc5db353bf0a51896e5dc5b820226.pkl does not exist, loading from directory embeddings/uuid_embeddings/


100%|██████████| 139637/139637 [03:58<00:00, 584.46it/s] 


Saving cache to embeddings/cache/6aadc5db353bf0a51896e5dc5b820226.pkl


In [24]:
tensor_dim = tensors[list(tensors.keys())[0]].shape[1]
tensor_dim

1024

In [25]:
zero_count = 0
value_count = 0
for k, vs in tqdm.tqdm(tensors.items()):
  for v in vs:
    zeros = np.count_nonzero(v)
    value_count += v.shape[0]
    zero_count += (v.shape[0] - zeros)

print(zero_count / value_count)
print(zero_count)
print(value_count)


100%|██████████| 139637/139637 [00:03<00:00, 39893.87it/s]

0.0
0
1379203072


In [26]:
all_chunks = torch.cat(list(tensors.values()))
all_chunks.shape


torch.Size([1346878, 1024])

In [27]:
# 3. Pool Tensors via VLAD
from cuml.cluster import KMeans
from cuml.decomposition import PCA

all_chunks_gpu = cp.array(all_chunks)

# 3.1 PCA Dim Reduction for chunks
pca = PCA(n_components=N_PCA_COMPONENTS)
pca.fit(all_chunks_gpu) # or a sample of it

reduced_chunks_gpu = pca.transform(all_chunks_gpu)

# 3.1 KMeans over all chunks to build codebook (PCA reduced)
# collect all chunks into flat array
kmeans = KMeans(n_clusters=N_CLUSTERS, n_init="auto", random_state=1)
kmeans.fit(reduced_chunks_gpu)

codebook = kmeans.cluster_centers_
cluster_labels = kmeans.labels_

In [28]:
codebook.shape

(64, 256)

In [29]:
from collections import defaultdict
# 3.2 VLAD Vectors from each track's chunk embeddings
vlad_vectors: Dict[str, cp.ndarray] = {}

codebook_gpu = cp.asarray(codebook)
cluster_labels_gpu = cp.asarray(cluster_labels)


current_idx = 0
for track_id, embeddings in tqdm.tqdm(tensors.items()):
  track_uuid = track_id.replace(".allchunks.pt", "")

  nframe = embeddings.shape[0]

  # 1. Get the chunk data and labels for this specific track
  # We slice the global arrays using the current offset
  track_data_reduced = reduced_chunks_gpu[current_idx : current_idx + nframe]
  track_labels_gpu = cluster_labels_gpu[current_idx : current_idx + nframe]
  
  # 2. Compute Residuals (Chunk - Assigned Centroid)
  # codebook_gpu[track_labels_gpu] gets the centroid vector for each chunk
  residuals = track_data_reduced - codebook_gpu[track_labels_gpu]
  
  # 3. Scatter Add (Aggregate residuals into clusters)
  # Create empty matrix (K clusters, D dimensions)
  vlad_mat = cp.zeros((N_CLUSTERS, N_PCA_COMPONENTS), dtype=cp.float32)
  # Add residuals to their corresponding cluster row
  cupyx.scatter_add(vlad_mat, track_labels_gpu, residuals)
  
  # 4. Power Normalization (sign(x) * sqrt(|x|))
  # This reduces the impact of visual/audio bursts (common in VLAD)
  vlad_mat = cp.sign(vlad_mat) * cp.sqrt(cp.abs(vlad_mat))
  
  # === FIX: INTRA-NORMALIZATION ===
  # L2 normalize EACH cluster (row) individually
  # This prevents the "Silence" cluster from dominating the vector
  # row_norms = cp.linalg.norm(vlad_mat, axis=1, keepdims=True)
  
  # # Avoid division by zero
  # row_norms[row_norms < 1e-12] = 1.0
  # vlad_mat = vlad_mat / row_norms
  
  # 5. Flatten to 1D vector
  vlad_vec = vlad_mat.flatten()
  
  # 6. Final Global L2 Normalization (Optional but recommended)
  norm = cp.linalg.norm(vlad_vec)
  if norm > 1e-12:
      vlad_vec /= norm
  
  # 7. Move to CPU
  # vlad_vectors[track_id] = cp.asnumpy(vlad_vec)
  # save immidiately
  np.save(f"embeddings/vlad_pooled/{track_uuid}.npy", vlad_vec)
  
  # Increment offset for next track
  current_idx += nframe


100%|██████████| 139637/139637 [11:29<00:00, 202.65it/s] 
